# Telco Customer Churn — Analysis & Prediction

**Author:** Jahanvi Kashyap  
**Dataset:** IBM Telco Customer Churn (Kaggle, ~7K rows)  
**Goal:** Identify *who* churns, *why*, and *what to do about it*.

---

## Notebook map
1. Business Framing  ← *you are here*
2. Data Loading & Inspection  ← *you are here*
3. Data Cleaning *(coming next)*
4. Exploratory Data Analysis
5. Cohort & Segmentation
6. Feature Engineering
7. Modeling (Logistic Regression + Random Forest)
8. Model Evaluation
9. Feature Importance + SHAP
10. Business Recommendations

## 1. Business Framing

### Why churn matters
Telecom companies live and die by **monthly recurring revenue (MRR)**. Every customer who cancels is a hole in next month's revenue — and acquiring a replacement is far more expensive.

Industry benchmarks (Harvard Business Review, Bain & Co.):
- Acquiring a new customer costs **5–25× more** than retaining an existing one.
- A **5% increase in retention** can lift profits by **25–95%**.
- Churned customers also stop generating word-of-mouth referrals — a hidden second cost.

### The economic model in one line
$$
\text{Net Saved Revenue} = (\text{customers retained}) \times (\text{ARPU}) \times (\text{remaining lifetime months}) \;-\; \text{retention offer cost}
$$

Where **ARPU** = Average Revenue Per User (here, `MonthlyCharges`).

### Why this matters for the model design
Two error types — and they are **not equally costly**:

| Error | What happens | Cost |
|---|---|---|
| **False Negative** (model says "won't churn" but does) | Customer leaves silently | Lost MRR forever — **HIGH cost** |
| **False Positive** (model says "will churn" but doesn't) | Loyal customer gets a discount they didn't need | Small wasted offer — **LOW cost** |

→ This asymmetry tells us to **optimize Recall**, not Accuracy. Better to over-flag than miss a leaver.

### Questions this analysis will answer
1. What is the overall churn rate, and how does it compare to industry norms (~22–25% telecom)?
2. Which customer segments churn most? (contract type, tenure, services bought)
3. Which features are the strongest predictors?
4. Can we predict churn well enough to act on it (target Recall ≥ 0.75)?
5. What concrete retention plays should the business run, and what is the expected ROI?

## 2. Data Loading & Inspection

**Goal of this section:** before any analysis, *look* at the data. Answer:
- How big is it?
- What columns and types?
- Any obvious mess (wrong dtype, missing values, duplicates)?
- What is the class balance of the target?

Skipping this step is the #1 reason analyses go wrong later.

In [ ]:
# ---------------------------------------------------------------
# Imports
# ---------------------------------------------------------------
# pandas      -> tables / dataframes (the analyst's spreadsheet)
# numpy       -> numeric arrays, used under the hood by pandas
# matplotlib  -> base plotting library
# seaborn     -> prettier statistical plots, built on matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display + style settings so output looks clean in the notebook
pd.set_option('display.max_columns', None)   # show all columns, never truncate
pd.set_option('display.width', 140)          # wider console output
sns.set_style('whitegrid')                   # subtle grid lines on plots
plt.rcParams['figure.figsize'] = (9, 5)      # default plot size

# Reproducibility: fix random seed so results are identical on every run
RANDOM_STATE = 42

In [ ]:
# ---------------------------------------------------------------
# Load the raw CSV
# ---------------------------------------------------------------
# Path is relative to this notebook's location (notebooks/).
# `..` means "go up one folder", then into data/raw.
DATA_PATH = '../data/raw/telco_churn.csv'

df = pd.read_csv(DATA_PATH)

# Show shape: (rows, columns). Quick sanity check vs Kaggle description (~7K rows, 21 cols).
print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')

In [ ]:
# First 5 rows — a visual sniff test.
# Watch for: weird values, all-zeros columns, IDs that look broken.
df.head()

**What to look for in `.head()`:**
- `customerID` — unique key, will be dropped before modeling.
- `SeniorCitizen` — already 0/1 (numeric), unlike most other Yes/No columns.
- Service columns use *three* states: `Yes`, `No`, `No internet service` / `No phone service`. We will collapse those during cleaning.
- `Churn` is `Yes/No` — our target.

In [ ]:
# ---------------------------------------------------------------
# Column types and non-null counts
# ---------------------------------------------------------------
# This is the MOST useful command in early EDA.
# We are looking for two red flags:
#   1. Columns that should be numeric but show 'object' (likely string mess inside)
#   2. Non-null counts < total rows (missing values)
df.info()

**Red flag spotted:** `TotalCharges` is `object` (string), but it should be numeric (it's a money amount).  
This is the famous Telco-dataset gotcha: 11 customers with `tenure = 0` (brand-new) have a literal space `" "` instead of a number. We will fix this in Section 3 (Cleaning).

Also notice: `.info()` reports zero nulls — but that is misleading because `" "` is a string, not `NaN`. **Missing data can hide as bad strings.** Always sanity-check.

In [ ]:
# Numeric summary — only the columns pandas thinks are numeric.
# Useful for spotting: impossible values, weird ranges, outliers.
df.describe()

In [ ]:
# Categorical summary — value counts for non-numeric columns.
# `include='object'` filters to text columns only.
df.describe(include='object')

In [ ]:
# ---------------------------------------------------------------
# Hunt for hidden missing values
# ---------------------------------------------------------------
# We confirmed TotalCharges has spaces. Let's quantify it explicitly.
blank_total_charges = (df['TotalCharges'].str.strip() == '').sum()
print(f'Rows with blank TotalCharges: {blank_total_charges}')

# Show those rows — what do they have in common?
df[df['TotalCharges'].str.strip() == ''][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

**Insight:** all 11 rows have `tenure = 0` → they are brand-new customers who haven't been billed yet, and none of them churned. This is *meaningful* missingness, not random. Two reasonable fixes (we'll decide in cleaning):
1. Drop them (only 0.16% of data, won't hurt).
2. Set `TotalCharges = 0` (logically correct: they have paid nothing yet).

In [ ]:
# ---------------------------------------------------------------
# Standard pandas null check — should be all zeros
# ---------------------------------------------------------------
df.isnull().sum().sort_values(ascending=False).head(5)

In [ ]:
# Duplicate rows? Duplicate customerIDs?
print(f'Duplicate rows:          {df.duplicated().sum()}')
print(f'Duplicate customerIDs:   {df["customerID"].duplicated().sum()}')

In [ ]:
# ---------------------------------------------------------------
# Target variable: class balance
# ---------------------------------------------------------------
# This single number drives modeling decisions later.
# Industry benchmark for telecom churn ~22-25%.
churn_counts = df['Churn'].value_counts()
churn_rate = df['Churn'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'count': churn_counts, 'percent': churn_rate.round(2)})
print(summary)
print(f"\nOverall churn rate: {churn_rate['Yes']:.2f}%")

In [ ]:
# Quick visual of class balance — useful for the README later.
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=df, x='Churn', order=['No', 'Yes'], palette=['#4C72B0', '#DD8452'], ax=ax)
ax.set_title(f'Customer Churn Distribution  (overall churn rate: {churn_rate["Yes"]:.1f}%)')
ax.set_ylabel('Number of customers')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
# Save the figure for the README — first artifact for the portfolio.
plt.savefig('../reports/figures/01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### Section 2 — Takeaways

1. **Size:** 7,043 customers × 21 columns. Healthy size for both EDA and modeling.
2. **Target:** ~26.5% churn rate. Consistent with telecom industry norms.
3. **Class imbalance:** moderate (roughly 3:1). Not severe enough to need SMOTE, but we will use **stratified train/test split** and watch **Recall** carefully.
4. **Data quality issues found:**
   - `TotalCharges` is stored as text and contains 11 blank values for tenure-0 customers. **→ fix in Section 3.**
   - Many service columns have a third "No internet/phone service" state that is logically the same as "No". **→ collapse in Section 3.**
5. **No duplicates.** `customerID` is a clean unique key.

**Next:** Section 3 — Data Cleaning. We will fix `TotalCharges`, normalize the categorical columns, and produce a clean dataframe saved to `data/processed/`.